Cell 1 — Setup & Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import sacrebleu

sns.set_theme(style="whitegrid")
df = pd.read_csv("../backend/data/medical_corpus.csv")
print(f"Loaded {len(df)} sentence pairs")
df.head()


Cell 2 — Data Engineering & Visualization
This satisfies "Show visualizations of the data used to train the ML model."

In [ ]:
# Domain distribution (matches your Table 5)
plt.figure(figsize=(9, 5))
df['domain_category'].value_counts().plot(kind='barh', color='teal')
plt.title("Sesotho-English Medical Corpus — Domain Distribution")
plt.xlabel("Number of Sentence Pairs")
plt.tight_layout()
plt.savefig("../docs/screenshots/domain_distribution.png")
plt.show()

# Sentence-length distribution (data engineering insight)
df['en_len'] = df['english_text'].str.split().apply(len)
df['st_len'] = df['sesotho_text'].str.split().apply(len)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['en_len'], ax=ax[0], color='steelblue', bins=20)
ax[0].set_title("English Phrase Length (words)")
sns.histplot(df['st_len'], ax=ax[1], color='darkorange', bins=20)
ax[1].set_title("Sesotho Phrase Length (words)")
plt.tight_layout()
plt.show()


Cell 3 — Model Architecture (NLLB-200)
This satisfies "Present the architecture of the ML model."

In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"  # lightweight, Colab-friendly

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Inspect architecture
print(model.config)        # Transformer encoder-decoder, attention heads, layers
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")


NLLB-200 (distilled-600M) is a Transformer encoder–decoder model. The encoder reads the source sentence into contextual embeddings using multi-head self-attention; the decoder generates the target tokens using cross-attention over the encoder output. Sesotho is the sot_Latn language code; English is eng_Latn.

Cell 4 — Inference Function

In [ ]:
def translate(text, src="eng_Latn", tgt="sot_Latn"):
    tokenizer.src_lang = src
    inputs = tokenizer(text, return_tensors="pt")
    out = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
        max_length=128
    )
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

translate("Take one tablet twice a day after meals.")


Cell 5 — Initial Performance Metrics
This satisfies "Share performance metrics... BLEU." Use BLEU + chrF++ + TER on your test split, exactly as your proposal promises.

In [ ]:
test = df.sample(min(50, len(df)), random_state=42)  # held-out sample
hypotheses, references = [], []

for _, row in test.iterrows():
    hyp = translate(row['english_text'], "eng_Latn", "sot_Latn")
    hypotheses.append(hyp)
    references.append(row['sesotho_text'])

bleu = sacrebleu.corpus_bleu(hypotheses, [references])
chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)  # chrF++
ter  = sacrebleu.corpus_ter(hypotheses, [references])

print(f"BLEU   : {bleu.score:.2f}")
print(f"chrF++ : {chrf.score:.2f}")
print(f"TER    : {ter.score:.2f}")
